# NeoCorpus — Foundation Corpus Preparation

Build a foundation pretraining corpus from
`karpathy/climbmix-400b-shuffle`.

## Pipeline

1. Configure the data source and collection settings.
2. Load a reference tokenizer for approximate token counting.
3. Stream and filter source documents.
4. Collect documents with resumable checkpoints.
5. Perform exact deduplication.
6. Create train, validation, and test splits.
7. Validate and save the final dataset.

## Notes

- Target: approximately 1.2B tokens.
- Reference tokenizer: HuggingFaceTB/SmolLM-135M.
- The NeoLM tokenizer is trained separately from scratch.
- Dataset publishing is handled in a separate notebook.

In [ ]:
from pathlib import Path
from transformers import AutoTokenizer
from huggingface_hub import HfApi
import json
import os
import shutil
from itertools import islice

import hashlib

from datasets import (
    Dataset,
    load_dataset,
    DatasetDict,
    Features,
    Value,
    load_from_disk,
)
from tqdm.auto import tqdm

In [ ]:
cwd = Path.cwd().resolve()

ROOT = next(
    (
        path for path in (cwd, *cwd.parents)
        if (path / "requirements.txt").is_file()
        and (path / "notebooks").is_dir()
    ),
    None,
)

if ROOT is None:
    raise RuntimeError("Project root not found.")

# Dataset source
SOURCE_ID = "karpathy/climbmix-400b-shuffle"
SOURCE_REVISION = "915333b4f8b8684f39aeaafea600fea6f43fb703"

SOURCE_SPLIT = "train"
TEXT_COLUMN = "text"

# Reference tokenizer (counting only)
COUNTING_TOKENIZER_ID = "HuggingFaceTB/SmolLM-135M"

# Collection
TOKEN_TARGET = 1_200_000_000
VAL_DOCS = 5_000
TEST_DOCS = 20_000
SEED = 1337

# Filtering
MIN_CHARS = 100
MAX_CHARS = 100_000

# Processing
BATCH_SIZE = 1_000

# Storage
DB_DIR = ROOT / "artifacts" / "pretrain_db"
COLLECTION_DIR = DB_DIR / "collection"
FINAL_DIR = DB_DIR / "final"

DB_DIR.mkdir(parents=True, exist_ok=True)

print("Root:", ROOT)
print("Source:", SOURCE_ID)
print("Target tokens:", f"{TOKEN_TARGET:,}")
print("Database:", DB_DIR)

In [ ]:
counting_tokenizer = AutoTokenizer.from_pretrained(
    COUNTING_TOKENIZER_ID,
    use_fast=True,
)

assert counting_tokenizer.is_fast

print("Tokenizer:", COUNTING_TOKENIZER_ID)
print("Vocabulary:", len(counting_tokenizer))
print("EOS:", counting_tokenizer.eos_token)
print("PAD:", counting_tokenizer.pad_token)

In [ ]:
api = HfApi()

source_info = api.dataset_info(
    SOURCE_ID,
    revision=SOURCE_REVISION,
)

source_files = sorted(
    file.rfilename
    for file in source_info.siblings
    if file.rfilename.endswith(".parquet")
)

print("Source:", SOURCE_ID)
print("Revision:", SOURCE_REVISION)
print("Parquet shards:", len(source_files))

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return None

    text = text.strip()

    if not (MIN_CHARS <= len(text) <= MAX_CHARS):
        return None

    # Exclude documents containing reserved special tokens
    special_tokens = (
        counting_tokenizer.eos_token,
        counting_tokenizer.pad_token,
    )

    if any(token in text for token in special_tokens if token):
        return None

    return text


def count_batch_tokens(texts):
    encodings = counting_tokenizer.backend_tokenizer.encode_batch(
        texts,
        add_special_tokens=False,
    )

    return [
        len(encoding.ids) + 1  # EOS per document
        for encoding in encodings
    ]

In [ ]:
samples = [
    "Hello world!",
    "   ",
    "A" * 150,
    "Transformers are neural network architectures. " * 5,
]

cleaned = [
    text
    for sample in samples
    if (text := clean_text(sample)) is not None
]

counts = count_batch_tokens(cleaned)

for text, count in zip(cleaned, counts):
    print(f"Chars: {len(text):,} | Tokens: {count:,}")

assert len(cleaned) == 2
assert all(count > 1 for count in counts)

print("Document cleaning & token counting: PASSED")

In [ ]:
CHUNK_DOCS = 10_000


def make_doc_id(shard_name, row_index):
    value = f"{SOURCE_REVISION}:{shard_name}:{row_index}"
    digest = hashlib.sha256(value.encode()).hexdigest()[:24]
    return f"neocorpus-{digest}"


def collect_corpus(
    output_dir=COLLECTION_DIR,
    train_target=TOKEN_TARGET,
    val_target=VAL_DOCS,
    test_target=TEST_DOCS,
    chunk_docs=CHUNK_DOCS,
):
    output_dir = Path(output_dir)
    chunks_dir = output_dir / "chunks"
    chunks_dir.mkdir(parents=True, exist_ok=True)

    # Prevent resuming with different settings
    config_path = output_dir / "collection_config.json"

    config = {
        "source": SOURCE_ID,
        "revision": SOURCE_REVISION,
        "counting_tokenizer": COUNTING_TOKENIZER_ID,
        "vocab_size": len(counting_tokenizer),
        "train_target": train_target,
        "val_target": val_target,
        "test_target": test_target,
        "min_chars": MIN_CHARS,
        "max_chars": MAX_CHARS,
        "chunk_docs": chunk_docs,
    }

    committed = sorted(chunks_dir.glob("chunk_*"))

    if config_path.exists():
        saved_config = json.loads(config_path.read_text())
        assert saved_config == config, "Collection config mismatch"
    else:
        assert not committed, "Chunks found without configuration"
        config_path.write_text(json.dumps(config, indent=2))

    # Discard only unfinished writes
    for path in chunks_dir.glob(".chunk_*.tmp"):
        shutil.rmtree(path)

    if committed:
        for i, path in enumerate(committed):
            assert path.name == f"chunk_{i:06d}"
            assert (path / "checkpoint.json").is_file()

        state = json.loads(
            (committed[-1] / "checkpoint.json").read_text()
        )
        assert state["chunk_index"] == len(committed)
    else:
        state = {
            "shard_index": 0,
            "row_index": 0,
            "train_tokens": 0,
            "train_docs": 0,
            "val_docs": 0,
            "test_docs": 0,
            "skipped_docs": 0,
            "chunk_index": 0,
        }

    def completed():
        return (
            state["train_tokens"] >= train_target
            and state["val_docs"] >= val_target
            and state["test_docs"] >= test_target
        )

    print("Starting state:", state)

    if completed():
        print("Collection already complete")
        return state

    rows = []

    def save_chunk():
        if not rows:
            return

        index = state["chunk_index"]

        final = chunks_dir / f"chunk_{index:06d}"
        temp = chunks_dir / f".chunk_{index:06d}.tmp"

        assert not final.exists()

        Dataset.from_list(rows).save_to_disk(str(temp))

        next_state = {
            **state,
            "chunk_index": index + 1,
        }

        with open(temp / "checkpoint.json", "w") as f:
            json.dump(next_state, f, indent=2)
            f.flush()
            os.fsync(f.fileno())

        temp.rename(final)

        state["chunk_index"] += 1
        rows.clear()

        print(
            f"Chunk {index:06d} | "
            f"Train tokens: {state['train_tokens']:,}"
        )

    for shard_index in range(
        state["shard_index"],
        len(source_files),
    ):
        shard_name = source_files[shard_index]

        source = load_dataset(
            SOURCE_ID,
            data_files=shard_name,
            revision=SOURCE_REVISION,
            split=SOURCE_SPLIT,
            streaming=True,
        )

        start_row = (
            state["row_index"]
            if shard_index == state["shard_index"]
            else 0
        )

        iterator = islice(enumerate(source), start_row, None)

        while batch := list(islice(iterator, BATCH_SIZE)):
            cleaned = [
                (i, clean_text(row[TEXT_COLUMN]))
                for i, row in batch
            ]

            valid_texts = [
                text for _, text in cleaned
                if text is not None
            ]

            counts = iter(count_batch_tokens(valid_texts))

            for row_index, text in cleaned:
                state["shard_index"] = shard_index
                state["row_index"] = row_index + 1

                if text is None:
                    state["skipped_docs"] += 1
                    continue

                token_count = next(counts)

                if state["val_docs"] < val_target:
                    split = "validation"
                    state["val_docs"] += 1

                elif state["test_docs"] < test_target:
                    split = "test"
                    state["test_docs"] += 1

                else:
                    split = "train"
                    state["train_docs"] += 1
                    state["train_tokens"] += token_count

                rows.append({
                    "doc_id": make_doc_id(shard_name, row_index),
                    "text": text,
                    "source": SOURCE_ID,
                    "source_shard": shard_name,
                    "token_count": token_count,
                    "split": split,
                })

                if len(rows) >= chunk_docs:
                    save_chunk()

                if completed():
                    save_chunk()
                    print("Collection complete")
                    return state

        state["shard_index"] = shard_index + 1
        state["row_index"] = 0
        save_chunk()

    raise RuntimeError("Source exhausted before reaching target")

In [ ]:
test_dir = DB_DIR / "collector_smoke"

test_state = collect_corpus(
    output_dir=test_dir,
    train_target=5_000,
    val_target=5,
    test_target=10,
    chunk_docs=100,
)

assert test_state["train_tokens"] >= 5_000
assert test_state["val_docs"] == 5
assert test_state["test_docs"] == 10

print("SMOKE TEST PASSED")
print(test_state)

In [ ]:
resumed_state = collect_corpus(
    output_dir=test_dir,
    train_target=5_000,
    val_target=5,
    test_target=10,
    chunk_docs=100,
)

assert resumed_state == test_state

print("RESUME CHECK PASSED")

In [ ]:
production_state = collect_corpus(
    output_dir=COLLECTION_DIR,
    train_target=TOKEN_TARGET,
    val_target=VAL_DOCS,
    test_target=TEST_DOCS,
)

print("Collection complete")
print("Train tokens:", f'{production_state["train_tokens"]:,}')
print("Train documents:", f'{production_state["train_docs"]:,}')
print("Chunks:", production_state["chunk_index"])

In [ ]:
chunk_paths = sorted(
    (COLLECTION_DIR / "chunks").glob("chunk_*")
)

assert chunk_paths, "No collected chunks found."
assert not FINAL_DIR.exists(), "Final dataset already exists."

FEATURES = Features({
    "doc_id": Value("string"),
    "text": Value("string"),
    "source": Value("string"),
    "source_shard": Value("string"),
    "token_count": Value("int64"),
})


def text_hash(text):
    return hashlib.sha256(text.encode("utf-8")).digest()


# Protect validation and test documents
holdout_hashes = set()

for path in tqdm(chunk_paths, desc="Indexing holdouts"):
    chunk = load_from_disk(str(path))

    for row in chunk:
        if row["split"] != "train":
            holdout_hashes.add(text_hash(row["text"]))


# Build deduplicated splits
def generate_split(split):
    seen = set()

    for path in chunk_paths:
        chunk = load_from_disk(str(path))

        for row in chunk:
            if row["split"] != split:
                continue

            if split == "train":
                digest = text_hash(row["text"])

                if digest in holdout_hashes or digest in seen:
                    continue

                seen.add(digest)

            yield {key: row[key] for key in FEATURES}


final_db = DatasetDict({
    split: Dataset.from_generator(
        generate_split,
        gen_kwargs={"split": split},
        features=FEATURES,
    )
    for split in ("train", "validation", "test")
})

final_db.save_to_disk(str(FINAL_DIR))

print(final_db)
print("Saved to:", FINAL_DIR)